### Rate-limit an agent’s tool calls

An agent can call the same tool repeatedly while investigating an issue. Implement a limiter that decides whether to allow each call.

def allow_calls(
    calls: list[tuple[int, str]], limit: int, window: int
) -> list[bool]:
    ...

Each input is (timestamp_in_seconds, tool_name). For each tool, allow at most limit calls in a rolling window of window seconds. 

At time t, a previous call counts if its timestamp is greater than t - window. A call exactly window seconds old has expired. Only allowed calls count, and each tool has its own limit. Timestamps arrive in nondecreasing order, and both limit and window are positive.

Example:
calls = [(0, "search"), (3, "search"), (9, "search"),
         (9, "lookup"), (10, "search")]
limit = 2
window = 10


To implement this rolling-window rate limiter efficiently, use collections.defaultdict(collections.deque) to track the timestamps of allowed calls per tool.

Because timestamps arrive in nondecreasing order, a deque lets you evict expired timestamps from the left in O(1) amortized time while appending new allowed timestamps to the right.

In [ ]:
from collections import defaultdict, deque

def allow_calls(
    calls: list[tuple[int, str]], limit: int, window: int
) -> list[bool]:
    history = defaultdict(deque)
    result = []

    for timestamp, tool in calls:
        q = history[tool]

        # Evict expired calls (a call counts only if ts > timestamp - window)
        while q and q[0] <= timestamp - window:
            q.popleft()

        # Check if we are under the limit for this tool
        if len(q) < limit:
            q.append(timestamp)
            result.append(True)
        else:
            result.append(False)

    return result


## Usage

calls = [(0, "search"), (3, "search"), (9, "search"),
         (9, "lookup"), (10, "search")]
limit = 2
window = 10




Interview Talking Points

• Why <= in the while loop? The prompt states a call counts if its timestamp is strictly greater than t - window, and a call exactly window seconds old has expired. Using q[0] <= timestamp - window ensures timestamp 0 is evicted at t = 10 (0 <= 10 - 10).
• Only allowed calls are recorded: Notice (9, "search") was rejected, so 9 was not appended to history["search"]. Recording rejected calls would unfairly block future valid requests.

• Complexity:
	• Time: O(N) where N is the number of calls. Every timestamp is pushed onto a deque at most once and popped at most once.
	• Space: O(min(N, T * limit)), where T is the number of unique tools, because each tool's deque never holds more than limit timestamps.

If the interviewer asks a follow-up like "What if timestamps arrive slightly out of order?" or "How would you clear out inactive tools so memory doesn't grow forever?", want to talk through how you'd adapt this?
